In [41]:
import pandas as pd
import re

PATH_DATA_RAW = '../data/raw/dataset.csv'
PATH_DATA_PROCESSED = '../data/processed/dataset_preprocessed.csv'

# Path untuk file kamus di folder src
PATH_KAMUS_BAKU_XLSX = '../src/kamuskatabaku.xlsx'
PATH_KAMUS_BAKU_CSV = '../src/kamus_kata_baku2.csv'
PATH_KAMUS_TEKNIS_CSV = '../src/kamus_teknis.csv'

In [43]:
# Load Dataset 
df = pd.read_csv(PATH_DATA_RAW, sep=';')

print("1. Informasi Dataset")
df.info() # Cukup panggil langsung, tidak perlu dibungkus print() lagi

print("\n")
print("2. Ukuran Dataset")
print(f"Jumlah Baris : {df.shape[0]} baris")
print(f"Jumlah Kolom : {df.shape[1]} kolom")
print(f"Nama Kolom   : {df.columns.tolist()}")

print("\n")
print("3. 10 Data teratas")
display(df.head(10))


1. Informasi Dataset
<class 'pandas.DataFrame'>
RangeIndex: 3329 entries, 0 to 3328
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   No      3329 non-null   int64
 1   Ulasan  3329 non-null   str  
dtypes: int64(1), str(1)
memory usage: 390.7 KB


2. Ukuran Dataset
Jumlah Baris : 3329 baris
Jumlah Kolom : 2 kolom
Nama Kolom   : ['No', 'Ulasan']


3. 10 Data teratas


,No,Ulasan
0,1,"Monitornya sangat bagus, untuk harga segini bi..."
1,2,Cepat akurat recommended
2,3,"admin sangat membantu, secepat kilat dalam mel..."
3,4,cakep batttt. mendarat dengan selamat semoga awet
4,5,"mantab, walaupun ga dipake buat game tp 165hrt..."
5,6,@
6,7,"barang bagus dan sesuai deskripsi, penanganan ..."
7,8,"Barang sampai dengan aman, kualitas bagus dan ..."
8,9,"Bagus pengiriman sangat cepat, rapih, toko san..."
9,10,"Untuk harganya oke banget monitornya, contrast..."


In [45]:
# Load Dataset Utama
df = pd.read_csv(PATH_DATA_RAW, sep=';')
print(f"Data awal di-load: {df.shape[0]} baris")

# Load Kamus Eksternal
df_kamus_xlsx = pd.read_excel(PATH_KAMUS_BAKU_XLSX)
# Ubah ke lowercase
dict_baku_xlsx = dict(zip(df_kamus_xlsx['tidak_baku'].astype(str).str.lower(), 
                          df_kamus_xlsx['kata_baku'].astype(str).str.lower()))

# Load Kamus Typo Manual 
df_kamus_csv = pd.read_csv(PATH_KAMUS_BAKU_CSV, sep=';')

# Kode di bawah ini sekarang akan berjalan lancar tanpa KeyError
dict_baku_csv = dict(zip(df_kamus_csv['Kata_Asal'].astype(str).str.lower(), 
                         df_kamus_csv['Kata_Baku'].astype(str).str.lower()))

# Load Kamus Teknis Monitor
df_kamus_teknis = pd.read_csv(PATH_KAMUS_TEKNIS_CSV, sep=';')
dict_teknis = dict(zip(df_kamus_teknis['Variasi_Penulisan'].astype(str).str.lower(), 
                       df_kamus_teknis['Penulisan_Standar'].astype(str)))

dict_normalisasi_baku = {**dict_baku_xlsx, **dict_baku_csv}

print(f"Total kamus kata baku (gabungan): {len(dict_normalisasi_baku)} kata")
print(f"Total kamus jargon teknis: {len(dict_teknis)} kata")

Data awal di-load: 3329 baris
Total kamus kata baku (gabungan): 4424 kata
Total kamus jargon teknis: 113 kata


In [46]:
import itertools

# Preview 10 contoh dari Kamus Kata Baku
print("Kamus Kata Baku")
sepuluh_baku = dict(itertools.islice(dict_normalisasi_baku.items(), 10))
for kata_salah, kata_bena in sepuluh_baku.items():
    print(f"{kata_salah} -> {kata_bena}")

print("\n")

# Preview 10 contoh dari Kamus Teknis
print("Kamus Teknis")
sepuluh_teknis = dict(itertools.islice(dict_teknis.items(), 10))
for variasi, standar in sepuluh_teknis.items():
    print(f"{variasi} -> {standar}")


Kamus Kata Baku
woww -> wow
aminn -> amin
met -> selamat
netaas -> menetas
keberpa -> keberapa
eeeehhhh -> eh
kata2nyaaa -> kata-katanya
hallo -> halo
kaka -> kakak
ka -> kak


Kamus Teknis
144hz -> 144Hz
144 hz -> 144Hz
144hert -> 144Hz
165hz -> 165Hz
165 hz -> 165Hz
100hz -> 100Hz
100 hz -> 100Hz
75hz -> 75Hz
75 hz -> 75Hz
60hz -> 60Hz


In [47]:
def text_preprocessing(text):
    if pd.isna(text):
        return ""
    
    # 1. Case Folding
    text = str(text).lower()
    
    # 2. Pembersihan Data
    # Menghapus karakter non-alfanumerik tapi memertahankan spasi
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    # Hapus multiple spasi menjadi single spasi
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 3. Normalisasi Data
    # Normalisasi Bahasa Teknis
    for variasi, standar in dict_teknis.items():
        pattern = r'\b' + re.escape(variasi) + r'\b'
        text = re.sub(pattern, standar, text)
        
    # Normalisasi Typo dan Kata Baku
    words = text.split()
    normalized_words = []
    
    for word in words:
        word_lower = word.lower()
        if word_lower in dict_normalisasi_baku:
            normalized_words.append(dict_normalisasi_baku[word_lower])
        else:
            normalized_words.append(word)
            
    final_text = ' '.join(normalized_words)
    return final_text

In [48]:
print("Memulai proses text preprocessing...")
df['Ulasan_Cleaned'] = df['Ulasan'].apply(text_preprocessing)

# Membuang baris kosong setelah dibersihkan
df = df[df['Ulasan_Cleaned'].str.strip() != '']
df = df.dropna(subset=['Ulasan_Cleaned']).reset_index(drop=True)

print(f"Selesai! Sisa data setelah cleaning: {df.shape[0]} baris.")

# Perbandingan Before/After
df_subset = df[['Ulasan', 'Ulasan_Cleaned']]
display(pd.concat([df_subset.head(5), df_subset.tail(5)]))

# Export data ke folder processed
df.to_csv(PATH_DATA_PROCESSED, index=False)
print(f"File berhasil disimpan di: {PATH_DATA_PROCESSED}")

Memulai proses text preprocessing...
Selesai! Sisa data setelah cleaning: 3326 baris.


,Ulasan,Ulasan_Cleaned
0,"Monitornya sangat bagus, untuk harga segini bi...",monitornya sangat bagus untuk harga segini bis...
1,Cepat akurat recommended,cepat akurat direkomendasikan
2,"admin sangat membantu, secepat kilat dalam mel...",admin sangat membantu secepat kilat dalam mela...
3,cakep batttt. mendarat dengan selamat semoga awet,cakep banget mendarat dengan selamat semoga awet
4,"mantab, walaupun ga dipake buat game tp 165hrt...",mantab walaupun tidak dipakai buat game tapi 1...
3321,"Info prodak standar, ngak add yng aneh, dan st...",info produk standar tidak ada yang aneh dan st...
3322,"Standar utk pengalaman blnja online, tpi kemah...",standar untuk pengalaman belanja online tapi k...
3323,"Biasaaaa ajah overall, tidak add yng spesial, ...",biasa saja overall tidak ada yang spesial saya...
3324,Blom add pengalaman retur jdi blum bsa menilai.,belum ada pengalaman retur jadi belum bisa men...
3325,Belom add pengalaman retur jdi blm bsa menilai...,belum ada pengalaman retur jadi belum bisa men...


File berhasil disimpan di: ../data/processed/dataset_preprocessed.csv


In [50]:
import os

# Persiapan data untuk uji kappa
ANNOTATION_DIR = '../data/annotations/'

os.makedirs(ANNOTATION_DIR, exist_ok=True)

# 100 data teratas dari dataframe yang sudah diproses
df_kappa_sample = df[['No', 'Ulasan_Cleaned']].head(100).copy()

df_kappa_sample.columns = ['No', 'Ulasan']

output_excel_path = os.path.join(ANNOTATION_DIR, 'data_uji_kappa.xlsx')
df_kappa_sample.to_excel(output_excel_path, index=False)

print(f"Berhasil mengekspor 100 data uji kappa ke:\n{output_excel_path}")

Berhasil mengekspor 100 data uji kappa ke:
../data/annotations/data_uji_kappa.xlsx


In [6]:
import os
import pandas as pd  

ANNOTATION_DIR = '../data/processed/'
os.makedirs(ANNOTATION_DIR, exist_ok=True)

df_hasil = pd.read_csv('../data/processed/dataset_preprocessed.csv') 

df_clean = df_hasil[['No', 'Ulasan_Cleaned']].copy()

df_clean.columns = ['No', 'Ulasan']

output_excel_path = os.path.join(ANNOTATION_DIR, 'data_clean.xlsx')
df_clean.to_excel(output_excel_path, index=False)

print("Proses selesai! File disimpan di:", output_excel_path)

Proses selesai! File disimpan di: ../data/processed/data_clean.xlsx
